# Experiment 6, follow-up (Notebook 06b): is the Kenya-to-Zambia gain real?

**Owner:** Michael (Workstream D)

**Why this notebook exists.** In Notebook 06, a model trained on the Kenyan people and tested on the Zambian people gained about **+0.03 AUROC** when the cough was added to the questions (MFCC and spectrogram summaries). Every other gain in Experiment 6 is zero within error. That result was not expected, it appears in one direction only, and it comes from a small training set (118 people, 84 with TB). After a Holm correction for the 21 gain tests of Notebook 06, the adjusted p-values are 0.059 (spectrogram) and 0.18 (MFCC), so neither passes the usual 0.05 line. These checks ask whether the result is stable. They do not decide whether to believe it.

## The three checks, and how each is read (fixed before the run)

| Check | What it does | If the gain is real | If it is chance or an artefact |
|---|---|---|---|
| **1. Other draws of Kenyan coughs** | Notebook 06 trained on one seeded draw of up to 10 coughs per person. Here we repeat the run with `N_DRAWS` other draws from a bigger pool of each person's coughs (up to `KENYA_POOL_CAP`) | Gain stays positive in most draws | Gain swings around zero from draw to draw |
| **2. Per Zambian phone and per site** | Same Kenya-trained model, scored on each Zambian phone (A, B, C) and each site separately | Gain appears on all three phones and both sites | Gain sits on one phone or one site only |
| **3. Shuffled-label control** | Train the cough model on the Kenyan people with their **TB labels shuffled** (the questions model keeps the real labels), `N_SHUFFLE` times, and compute the gain each time | The observed gain is larger than almost all shuffled gains (empirical p below 0.05) | The observed gain falls inside the shuffled spread |

**Reading rules, fixed before the run** (they are applied in code in Step 12):

* **Check 1:** *stable* if the gain is above zero in at least 8 of the 10 draws; *draw-specific* if it is zero or below in a third or more of the draws; otherwise *mixed*.
* **Check 2:** *general* if the gain is above zero on all three phones and at both sites; *absent* if it is above zero on none; otherwise *local*.
* **Check 3:** the gain *beats the shuffle* if its empirical p is below 0.05.
* **Supported** only if check 1 is stable **and** check 3 beats the shuffle. Check 2 says where the gain lives.

Even a supported gain says only that the cough score carries label information between these two cohorts. It does not show that the signal is TB itself, because differences between the cohorts could contribute.

## Before you press Run: Kaggle settings

1. **Add data:** only your Zambia dataset `zambia-outputs-michael` (the bundles from Notebook 01). **No CODA outputs are needed.** Do not attach the 84 GB Zambia source dataset.
2. **Secret:** `AZURE_STORAGE_CONNECTION_STRING`, attached to the notebook (same as before).
3. **Accelerator:** GPU T4 x2 (used only for the VGGish step). **Internet: ON**.
4. Keep the notebook **private**. Exports contain aggregate tables and figures only.
5. Use **Save & Run All (Commit)**, not an interactive session, and set persistence to **No persistence**.

## What this notebook does

| Step | What |
|---|---|
| 0 to 1b | Setup, connect to the data, audio helpers (as in Notebook 06) |
| 3 | Zambia: cohort and 0.5 s clips (as in Notebook 06) |
| 4 | Kenya: the original draw of passive coughs (as in Notebook 06) |
| 5 | **New:** a bigger pool of Kenyan coughs per person, and features for every clip |
| 6 | The same three representations for every clip |
| 7 | Model functions (as in Notebook 06) |
| 8 | Reproduce the Notebook 06 Kenya-to-Zambia gain with the original draw |
| 9 | **Check 1:** other draws |
| 10 | **Check 2:** per phone and per site |
| 11 | **Check 3:** shuffled-label control |
| 12 | Save and package the results |

**Data protection.** Nothing derived from audio leaves the session except aggregate tables and figures. No Zambian or Kenyan label is used for anything but scoring, and the Kenyan labels are used for training only, exactly as in Notebook 06.

## Step 0. Setup

**What we do.** Import libraries and set every parameter in one place.

**Choices fixed in advance (not tuned on any test result).**

* `KENYA_DEVICE = "pixel"`: TBscreen recorded each cough on a phone and two microphones. We use the **phone**, because CODA's app and the Zambia phones are phones. Change it to `"yeti"` or `"codec"` only as a labelled sensitivity run.
* `KENYA_MAX_PER_PARTICIPANT = 10`: the cap from the plan (about 1,500 files). Coughs are picked by a seeded random draw.
* Zambia is scored **per phone (A, B, C)** and **pooled** (every clip of a person, all phones, averaged).

In [ ]:
import os, io, re, sys, json, time, zipfile, subprocess, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import soundfile as sf, librosa
from scipy.stats import norm
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score
from IPython.display import display
warnings.filterwarnings("ignore")

TEST_MODE = os.environ.get("EXT_TEST_MODE") == "1"
NB_VERSION = "nb06b-v1"
CFG = dict(
    CODA_CONTAINER = "coda",      CODA_PREFIX = "v1/Train/",
    TB_CONTAINER   = "tbscreen",  TB_PREFIX   = "v1/",
    LOCAL_ROOT     = None,        # offline testing only: a folder with one sub-folder per container
    # attached inputs (None = search INPUT_ROOT)
    INPUT_ROOT         = "/kaggle/input",
    ZAMBIA_INPUT       = None,    # folder or zip holding the Notebook 01 bundles
    CODA_FOLDS_INPUT   = None,    # folder holding coda_folds.csv from Notebook 04
    CODA_CACHE_INPUT   = None,    # folder holding coda_wave_*.npy / coda_index_*.csv from Notebook 04
    WORK_DIR           = "/kaggle/working",
    # data choices
    CLIP_SECONDS       = 0.5,
    MIN_DETECTOR_SCORE = 0.0,     # CODA: optionally drop coughs the detector was less sure about
    MIN_TB_PER_GROUP   = 15,      # CODA held-out groups (only used if coda_folds.csv is not found)
    KENYA_DEVICE       = "pixel",
    KENYA_MAX_PER_PARTICIPANT = 10,
    ZAMBIA_PHONES      = ["Phone A", "Phone B", "Phone C"],
    # models (identical to Notebook 04)
    SENSITIVITY_TARGET = 0.90,
    N_FOLDS            = 5,
    INNER_K            = 4,
    PROBE_C            = 0.1,
    CLIN_C             = 1.0,
    MAX_MISSINGNESS_GAP = 0.20,   # drop a question if its missingness differs by more than this between TB groups
    DOWNLOAD_WORKERS   = 32,
    N_BOOT             = 1000,
    PROBE_MAX_CLIPS    = 10,      # nine-country probe: at most this many clips per person
    RUN_OPTIONAL_REVERSE = False, # (unused here)
    KENYA_POOL_CAP     = 30,      # Kenya: coughs per person in the pool that the other draws are taken from
    N_DRAWS            = 10,      # check 1: number of other draws
    N_SHUFFLE          = 100,     # check 3: number of shuffled-label runs per representation
    SEED               = 42,
)
if TEST_MODE:
    CFG.update(LOCAL_ROOT="/home/claude/ext_test/fixture_stores", ZAMBIA_INPUT="/home/claude/ext_test/zambia_export",
               INPUT_ROOT="/nonexistent", WORK_DIR="/home/claude/ext_test/work", N_BOOT=100, MIN_TB_PER_GROUP=4,
               KENYA_MAX_PER_PARTICIPANT=4, RUN_OPTIONAL_REVERSE=False, KENYA_POOL_CAP=8, N_DRAWS=3, N_SHUFFLE=4)

WORK = Path(CFG["WORK_DIR"]); OUT = WORK / "outputs_external"; CACHE = WORK / "cache_external"
RES = OUT / "results" / "e6_external"; FIG_DIR = OUT / "figures" / "external"
for d in (OUT, CACHE, RES, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CFG["SEED"])
sns.set_theme(style="whitegrid"); plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

try:
    import torch, torch.nn as nn
    TORCH_OK, DEVICE = True, ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(CFG["SEED"])
except ImportError:
    TORCH_OK, DEVICE = False, "cpu"
print(f"Test mode: {TEST_MODE} | PyTorch: {TORCH_OK} | device: {DEVICE}")

## Step 1. Connect to the data

**What we do.** One small wrapper opens an Azure container (or a local folder when testing offline). `fetch_many` downloads many small files in parallel and retries failures. The same wrapper opens `coda` and `tbscreen`.

In [ ]:
class LocalStore:
    # A folder that mirrors the container; blob names are paths relative to it.
    def __init__(self, root):
        self.root = Path(root)
    def walk(self, prefix):
        p = self.root / prefix
        dirs = [f"{prefix}{c.name}/" for c in sorted(p.iterdir()) if c.is_dir()] if p.is_dir() else []
        files = {f"{prefix}{c.name}": c.stat().st_size for c in sorted(p.iterdir()) if c.is_file()} if p.is_dir() else {}
        return dirs, files
    def list(self, prefix):
        p = self.root / prefix
        for f in sorted(p.rglob("*")) if p.is_dir() else []:
            if f.is_file():
                yield f.relative_to(self.root).as_posix(), f.stat().st_size
    def get(self, name):
        return (self.root / name).read_bytes()

class AzureStore:
    def __init__(self, conn, container):
        from azure.storage.blob import BlobServiceClient
        self.c = BlobServiceClient.from_connection_string(conn).get_container_client(container)
    def walk(self, prefix):
        dirs, files = [], {}
        for item in self.c.walk_blobs(name_starts_with=prefix, delimiter="/"):
            if item.name.endswith("/"):
                dirs.append(item.name)
            else:
                files[item.name] = item.size
        return dirs, files
    def list(self, prefix):
        for b in self.c.list_blobs(name_starts_with=prefix):
            if b.size:
                yield b.name, b.size
    def get(self, name):
        return self.c.download_blob(name, max_concurrency=1).readall()

def azure_connection_string():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("AZURE_STORAGE_CONNECTION_STRING")
    except Exception:
        return os.environ["AZURE_STORAGE_CONNECTION_STRING"]

if not CFG["LOCAL_ROOT"]:
    try:
        import azure.storage.blob  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "azure-storage-blob"], check=True)
    _conn = azure_connection_string()

def open_store(container):
    if CFG["LOCAL_ROOT"]:
        return LocalStore(Path(CFG["LOCAL_ROOT"]) / container)
    return AzureStore(_conn, container)

STORE_CODA, STORE_TB = open_store(CFG["CODA_CONTAINER"]), open_store(CFG["TB_CONTAINER"])

def fetch_many(store, names, decode, workers=None, label="files", tries=4):
    # Download many small blobs in parallel; returns {name: decoded or None} and reports failures.
    workers = workers or CFG["DOWNLOAD_WORKERS"]
    out, failed, t0, done = {}, [], time.time(), 0
    def one(n):
        for k in range(tries):
            try:
                return n, decode(store.get(n))
            except Exception as exc:
                if k == tries - 1:
                    return n, exc
                time.sleep(1.5 ** k)
    with ThreadPoolExecutor(workers) as pool:
        futs = [pool.submit(one, n) for n in names]
        step = max(1, len(futs) // 10)
        for f in as_completed(futs):
            n, v = f.result(); done += 1
            if isinstance(v, Exception):
                failed.append((n, repr(v)[:200])); out[n] = None
            else:
                out[n] = v
            if done % step == 0 or done == len(futs):
                print(f"  {label}: {done:,}/{len(futs):,} in {time.time() - t0:.0f} s", flush=True)
    if failed:
        print(f"  {len(failed)} {label} failed, e.g. {failed[:2]}")
    return out

def find_first(name, roots):
    # first file called `name` under any of the given folders (None entries are skipped)
    for r in roots:
        if r and Path(r).exists():
            hits = sorted(Path(r).rglob(name))
            if hits:
                return hits[0]
    return None
print("Connected to:", CFG["CODA_CONTAINER"], "and", CFG["TB_CONTAINER"], "(local test folders)" if CFG["LOCAL_ROOT"] else "")

## Step 1b. Audio front end and modelling helpers

**What we do.** Load, unchanged, the functions of Notebook 04: the VGGish input front end, `to_clip` (the loudest 0.5 s window at 16 kHz, used for **every** dataset), the MFCC and VGGish extractors, DeLong's test, bootstrap intervals, the frozen-threshold rule and the logistic-regression helpers. Using the same code is what makes the three datasets comparable.

In [ ]:
# VGGish input front end, ported from Google's AudioSet mel_features.py / vggish_input.py
# (Apache License 2.0), exactly as used to produce TBscreen's stored features.
import numpy as np
import librosa

VGG_SR, VGG_WIN_S, VGG_HOP_S = 16000, 0.025, 0.010
VGG_MELS, VGG_FMIN, VGG_FMAX, VGG_LOG_OFFSET = 64, 125.0, 7500.0, 0.01
VGG_FRAMES = 96


def _frame(data, window_length, hop_length):
    n = 1 + int(np.floor((data.shape[0] - window_length) / hop_length))
    if n < 1:
        raise ValueError(f"signal too short: {data.shape[0]} samples < window {window_length}")
    shape = (n, window_length) + data.shape[1:]
    strides = (data.strides[0] * hop_length,) + data.strides
    return np.lib.stride_tricks.as_strided(data, shape=shape, strides=strides)


def _periodic_hann(n):
    return 0.5 - 0.5 * np.cos(2 * np.pi / n * np.arange(n))


def _hz_to_mel(f):
    return 1127.0 * np.log(1.0 + f / 700.0)


def _mel_matrix(num_mel_bins, num_spectrogram_bins, sr, lo, hi):
    nyq = sr / 2.0
    bins_mel = _hz_to_mel(np.linspace(0.0, nyq, num_spectrogram_bins))
    edges = np.linspace(_hz_to_mel(lo), _hz_to_mel(hi), num_mel_bins + 2)
    m = np.empty((num_spectrogram_bins, num_mel_bins))
    for i in range(num_mel_bins):
        l, c, u = edges[i:i + 3]
        m[:, i] = np.maximum(0.0, np.minimum((bins_mel - l) / (c - l), (u - bins_mel) / (u - c)))
    m[0, :] = 0.0
    return m


def log_mel(data, sr=VGG_SR):
    win = int(round(sr * VGG_WIN_S)); hop = int(round(sr * VGG_HOP_S))
    fft = 2 ** int(np.ceil(np.log(win) / np.log(2.0)))
    frames = _frame(np.ascontiguousarray(data, dtype=np.float64), win, hop)
    spec = np.abs(np.fft.rfft(frames * _periodic_hann(win), fft))
    mel = spec @ _mel_matrix(VGG_MELS, spec.shape[1], sr, VGG_FMIN, VGG_FMAX)
    return np.log(mel + VGG_LOG_OFFSET)


def vggish_examples(wave, sr):
    """Waveform -> array (n_examples, 96, 64), as vggish_input.waveform_to_examples."""
    wave = np.asarray(wave, dtype=np.float64)
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    if sr != VGG_SR:
        wave = librosa.resample(wave, orig_sr=sr, target_sr=VGG_SR)
    lm = log_mel(wave, VGG_SR)
    return np.ascontiguousarray(_frame(lm, VGG_FRAMES, VGG_FRAMES), dtype=np.float32)


def vggish_example_padded(wave, sr, seconds=1.0):
    """Centre a short clip in `seconds` of zeros (as TBscreen did) and return one (96, 64) example."""
    wave = np.asarray(wave, dtype=np.float64)
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    if sr != VGG_SR:
        wave = librosa.resample(wave, orig_sr=sr, target_sr=VGG_SR)
    n = int(round(VGG_SR * seconds))
    wave = librosa.util.pad_center(wave[:n], size=n, mode="constant") if len(wave) < n else wave[:n]
    return vggish_examples(wave, VGG_SR)[0]

In [ ]:
CLIP_N = int(round(VGG_SR * CFG["CLIP_SECONDS"]))
def to_clip(b):
    w, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=True)
    w = w.mean(axis=1)
    dur = len(w) / sr
    if sr != VGG_SR:
        w = librosa.resample(w, orig_sr=sr, target_sr=VGG_SR)
    if len(w) >= CLIP_N:                             # keep the loudest CLIP_N window (CODA clips are already 0.5 s)
        e = np.convolve(w ** 2, np.ones(CLIP_N), "valid"); s = int(np.argmax(e)); w = w[s:s + CLIP_N]
    else:
        w = np.pad(w, ((CLIP_N - len(w)) // 2, CLIP_N - len(w) - (CLIP_N - len(w)) // 2))
    return w.astype(np.float32), sr, dur

In [ ]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-4, 1 - 1e-4); return np.log(p / (1 - p))

def lr_pipe(C, impute=False):
    steps = ([SimpleImputer(strategy="median")] if impute else []) + [StandardScaler(),
             LogisticRegression(C=C, max_iter=5000, class_weight="balanced")]
    return make_pipeline(*steps)

def probe(Xtr, ytr, Xte):
    return lr_pipe(CFG["PROBE_C"]).fit(Xtr, ytr).predict_proba(Xte)[:, 1]

def part_mean(subjects, p):
    return pd.Series(np.asarray(p, dtype=float)).groupby(np.asarray(subjects)).mean()

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    out = np.empty(N); out[J] = T; return out

def delong(y, p_ref, p_new):
    y = np.asarray(y).astype(int); pos = y == 1; m, n = pos.sum(), (~pos).sum()
    aucs, V10, V01 = [], [], []
    for p in (p_ref, p_new):
        p = np.asarray(p, dtype=float); X, Y = p[pos], p[~pos]
        tx, ty, tz = _midrank(X), _midrank(Y), _midrank(np.r_[X, Y])
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        V10.append((tz[:m] - tx) / n); V01.append(1 - (tz[m:] - ty) / m)
    S = np.cov(np.vstack(V10)) / m + np.cov(np.vstack(V01)) / n
    se = np.sqrt(max(S[0, 0] + S[1, 1] - 2 * S[0, 1], 1e-12)); d = aucs[1] - aucs[0]
    return dict(diff=d, lo=d - 1.96 * se, hi=d + 1.96 * se, p=float(2 * norm.sf(abs(d / se))))

def boot_ci(y, p):
    y, p = np.asarray(y), np.asarray(p); r = np.random.default_rng(CFG["SEED"]); out = []
    for _ in range(CFG["N_BOOT"]):
        i = r.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p[i]))
    return np.percentile(out, [2.5, 97.5])

def threshold_for(y, p, sens):
    # Highest threshold that still catches `sens` of the TB cases in the training predictions.
    pos = np.sort(np.asarray(p)[np.asarray(y) == 1])
    return pos[min(len(pos) - 1, int(np.floor((1 - sens) * len(pos))))] if len(pos) else 0.5

def run_protocol(P, clips, X, folds, clin_cols, inner_k=4):
    # For each test fold: train B (questions), audio, and A (questions + out-of-fold audio score) on the rest.
    # Thresholds for the target sensitivity are set on TRAINING predictions only (frozen), then applied to the test fold.
    rows = []
    for f in sorted(pd.unique(folds)):
        te, tr = P.index[folds == f], P.index[folds != f]
        ctr, cte = clips.participant.isin(tr).values, clips.participant.isin(te).values
        a_te = part_mean(clips.participant.values[cte], probe(X[ctr], clips.label.values[ctr], X[cte])).reindex(te)
        a_tr = pd.Series(np.nan, index=tr)
        k = min(inner_k, int(P.loc[tr, "label"].value_counts().min()))
        for itr, iva in StratifiedKFold(k, shuffle=True, random_state=CFG["SEED"]).split(tr, P.loc[tr, "label"]):
            mt, mv = clips.participant.isin(tr[itr]).values, clips.participant.isin(tr[iva]).values
            a_tr.update(part_mean(clips.participant.values[mv], probe(X[mt], clips.label.values[mt], X[mv])))
        ytr = P.loc[tr, "label"].values
        res = pd.DataFrame({"label": P.loc[te, "label"].values, "fold": f, "audio": a_te.values}, index=te)
        thr = {"audio": threshold_for(ytr, a_tr.values, CFG["SENSITIVITY_TARGET"])}
        for name, cols in (("B", clin_cols), ("A", None)):
            Xtr = P.loc[tr, clin_cols].values if name == "B" else np.c_[P.loc[tr, clin_cols].values, logit(a_tr.values)]
            Xte = P.loc[te, clin_cols].values if name == "B" else np.c_[P.loc[te, clin_cols].values, logit(a_te.values)]
            m = lr_pipe(CFG["CLIN_C"], impute=True)
            oof = cross_val_predict(m, Xtr, ytr, cv=StratifiedKFold(k, shuffle=True, random_state=CFG["SEED"]), method="predict_proba")[:, 1]
            thr[name] = threshold_for(ytr, oof, CFG["SENSITIVITY_TARGET"])
            res[name] = m.fit(Xtr, ytr).predict_proba(Xte)[:, 1]
        for name in ("B", "audio", "A"):
            res[f"{name}_flag"] = (res[name] >= thr[name]).astype(int)
        rows.append(res)
    return pd.concat(rows)

def sens_spec(y, flag):
    y, flag = np.asarray(y), np.asarray(flag)
    return (flag[y == 1].mean() if (y == 1).any() else np.nan), (1 - flag[y == 0].mean() if (y == 0).any() else np.nan)

def summarise(pp, setting):
    rows = []
    for m, lab in (("B", "Questions only (B)"), ("audio", "Cough only"), ("A", "Questions + cough (A)")):
        lo, hi = boot_ci(pp.label, pp[m]); se, sp = sens_spec(pp.label, pp[f"{m}_flag"])
        rows.append({"setting": setting, "model": lab, "AUROC": round(roc_auc_score(pp.label, pp[m]), 3),
                     "95% CI": f"{lo:.2f} to {hi:.2f}", "sensitivity at frozen threshold": round(se, 2),
                     "specificity at frozen threshold": round(sp, 2)})
    d = delong(pp.label, pp.B, pp.A)
    rows.append({"setting": setting, "model": "Gain Δ = A − B", "AUROC": round(d["diff"], 3),
                 "95% CI": f"{d['lo']:+.2f} to {d['hi']:+.2f}", "sensitivity at frozen threshold": f"p = {d['p']:.2f}",
                 "specificity at frozen threshold": ""})
    return pd.DataFrame(rows)

In [ ]:
def mfcc_summary(W):
    out = []
    for w in W:
        m = librosa.feature.mfcc(y=w, sr=VGG_SR, n_mfcc=20, n_fft=400, hop_length=160)
        out.append(np.r_[m.mean(axis=1), m.std(axis=1)])
    return np.asarray(out, dtype=np.float32)



VGGISH_URL = "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth"
if TORCH_OK:
    class VGGishBackbone(nn.Module):
        def __init__(self):
            super().__init__()
            def conv(cin, cout):
                return [nn.Conv2d(cin, cout, 3, 1, 1), nn.ReLU(inplace=True)]
            self.features = nn.Sequential(*conv(1, 64), nn.MaxPool2d(2, 2), *conv(64, 128), nn.MaxPool2d(2, 2),
                                          *conv(128, 256), *conv(256, 256), nn.MaxPool2d(2, 2),
                                          *conv(256, 512), *conv(512, 512), nn.MaxPool2d(2, 2))
            self.embeddings = nn.Sequential(nn.Linear(512 * 24, 4096), nn.ReLU(True), nn.Linear(4096, 4096), nn.ReLU(True),
                                            nn.Linear(4096, 128), nn.ReLU(True))
        def forward(self, x):
            x = self.features(x).permute(0, 2, 3, 1).contiguous()
            return self.embeddings(x.view(x.size(0), -1))
    def vggish_embed(X, batch=256):
        wpath = WORK / "vggish-10086976.pth"
        if not wpath.exists():
            torch.hub.download_url_to_file(VGGISH_URL, str(wpath), hash_prefix="10086976")
        try:
            sd = torch.load(wpath, map_location="cpu", weights_only=True)
        except TypeError:
            sd = torch.load(wpath, map_location="cpu")
        bb = VGGishBackbone(); bb.load_state_dict(sd, strict=True); bb.eval().to(DEVICE)
        out = []
        with torch.no_grad():
            for i in range(0, len(X), batch):
                out.append(bb(torch.as_tensor(X[i:i + batch], device=DEVICE)[:, None]).cpu().numpy())
        return np.concatenate(out).astype(np.float32)

## Step 3. Zambia: the first test country

**What we do.** Read the Notebook 01 bundles (cohort, harmonised questions, segmented coughs), keep the **analysis cohort** (Chawama and Kanyama, Phones A, B and C, with a TB label), and apply the **same `to_clip`** as for CODA to every segmented cough: the loudest 0.5 s at 16 kHz. This replaces the old one-second padded input.

**What to look for.** About 325 people, 57 with TB, and about 5,287 clips. Small differences from these numbers are expected if you re-ran segmentation; large ones mean something went wrong upstream.

**Question definitions differ across datasets** (a limitation to report): smoking is *smoked in the last week* in CODA but *current or former smoker* in Zambia; cough duration is a day count in CODA and a category midpoint in Zambia. Models trained on CODA see Zambia's version of each answer.

In [ ]:
def find_input(marker, zip_hint, name, roots):
    # Return a folder containing `marker`, extracting a matching zip into WORK/restored/<name> if needed.
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        hits = sorted(p for p in root.rglob(marker) if "restored" not in p.parts)
        if hits:
            return hits[0].parent
        for zp in sorted(root.rglob(f"*{zip_hint}*.zip")):
            with zipfile.ZipFile(zp) as z:
                if any(n == marker or n.endswith("/" + marker) for n in z.namelist()):
                    dest = WORK / "restored" / name
                    if not any(dest.rglob(marker)):
                        print(f"Extracting {zp.name} ...")
                        z.extractall(dest)
                    return next(dest.rglob(marker)).parent
    return None


Z_ROOTS = [CFG["ZAMBIA_INPUT"]] if CFG["ZAMBIA_INPUT"] else [CFG["INPUT_ROOT"]]

def find_seg_dir(roots):
    # A usable folder holds segmentation_manifest.csv AND every clip it lists. A partial copy (for example a stray,
    # half-listed segmented_coughs folder in the dataset) is skipped, and the zip bundle is used instead.
    for root in roots:
        if not (root and Path(root).exists()):
            continue
        for m in sorted(Path(root).rglob("segmentation_manifest.csv")):
            seg = m.parent / "segmented_coughs"
            if "restored" in m.parts or not seg.is_dir():
                continue
            n_rows = sum(1 for _ in open(m)) - 1
            n_wav = sum(1 for _ in seg.rglob("*.wav"))
            if n_wav >= n_rows:
                return m.parent
            print(f"Skipping {m.parent}: only {n_wav:,} of {n_rows:,} clips are there")
    for root in roots:
        if not (root and Path(root).exists()):
            continue
        for zp in sorted(Path(root).rglob("*segmented*.zip")):
            with zipfile.ZipFile(zp) as z:
                if any(n == "segmentation_manifest.csv" or n.endswith("/segmentation_manifest.csv") for n in z.namelist()):
                    dest = WORK / "restored" / "zambia_segmented"
                    if not any(dest.rglob("segmentation_manifest.csv")):
                        print(f"Extracting {zp.name} ...")
                        z.extractall(dest)
                    return next(dest.rglob("segmentation_manifest.csv")).parent
    return None

z_seg_dir = find_seg_dir(Z_ROOTS)
z_meta_dir = find_input("clinical_harmonised.csv", "metadata", "zambia_metadata", Z_ROOTS)
assert z_seg_dir is not None, "A complete zambia_segmented_coughs bundle was not found: attach your Zambia dataset (see the settings above)"
assert z_meta_dir is not None, "zambia_metadata_cohort bundle not found: attach your Zambia dataset (see the settings above)"
Z_SEG_ROOT = z_seg_dir / "segmented_coughs"

z_clin = pd.read_csv(z_meta_dir / "clinical_harmonised.csv").set_index("subject_id")
z_cohort = pd.read_csv(z_meta_dir / "cohort_reproduction.csv").set_index("subject_id")
z_manifest = pd.read_csv(z_seg_dir / "segmentation_manifest.csv")

SHARED9 = ["age", "male", "smoker", "hiv_pos", "prev_tb", "cough_days", "fever", "weight_loss", "night_sweats"]
ZP = z_clin.loc[z_clin.index.intersection(z_cohort.index), ["label", "site"] + SHARED9].copy()
ZP = ZP[ZP.label.notna()]; ZP["label"] = ZP.label.astype(int)
zm = z_manifest[z_manifest.subject_id.isin(ZP.index) & z_manifest.device.isin(CFG["ZAMBIA_PHONES"])]
zm = zm.sort_values(["subject_id", "device", "clip_idx"]).reset_index(drop=True)

absent = [r for r in zm.rel_path if not (Z_SEG_ROOT / r).exists()]
assert not absent, f"{len(absent):,} cohort clips are missing from {Z_SEG_ROOT}, e.g. {absent[:2]}"
def read_clip(rel):
    return to_clip((Z_SEG_ROOT / rel).read_bytes())
t0 = time.time()
with ThreadPoolExecutor(8) as pool:
    zres = list(pool.map(read_clip, zm.rel_path))
print(f"Cut {len(zres):,} Zambia clips in {time.time() - t0:.0f} s")
ZWAVE = np.stack([r[0] for r in zres])
ZC = pd.DataFrame({"participant": zm.subject_id.values, "phone": zm.device.values, "clip_idx": zm.clip_idx.values,
                   "orig_seconds": [round(r[2], 3) for r in zres]})
ZC["label"] = ZC.participant.map(ZP.label).astype(int)
ZP = ZP.loc[ZP.index.isin(ZC.participant)]
print(f"Zambia: {len(ZP)} people ({int(ZP.label.sum())} TB), {len(ZC):,} clips | people per phone:",
      ZC.groupby("phone").participant.nunique().to_dict())
display(ZP.groupby("site").label.agg(people="size", TB="sum"))

## Step 4. Kenya (TBscreen): the second test country

**What we do.** Map the container, read `Metadata.csv`, harmonise the nine questions with Zhang et al.'s published mapping (as in Notebook 03), and list the **passive coughs** of one device (`KENYA_DEVICE`). For every person we keep a seeded random draw of up to `KENYA_MAX_PER_PARTICIPANT` coughs, download them, and apply `to_clip`.

**What to look for.** About 148 people (102 TB, 46 non-TB). **Kenya is case-control** (TB against other respiratory illness), so its absolute AUROC is not comparable with a screening population: read the gain Δ and the threshold transfer. In Kenya, HIV status is missing for most non-TB people; the missingness rule removes any question whose missingness differs by more than 20 points between the two groups, so a model cannot learn "missing means no TB".

In [ ]:
def _norm(s):
    return re.sub(r"[\s_\-]+", "", s.lower())
LEAF_DIRS = {"audiofiles", "scalogramnumpy", "vggishembeddings", "pytorchembeddings"}
def map_layout(store, prefix, max_depth=5):
    found_dirs, small_files, queue = [], {}, [(prefix, 0)]
    while queue:
        p, d = queue.pop(0)
        dirs, files = store.walk(p)
        small_files.update(files)
        for sd in dirs:
            found_dirs.append(sd)
            if _norm(Path(sd.rstrip("/")).name) not in LEAF_DIRS and d < max_depth:
                queue.append((sd, d + 1))
    return found_dirs, small_files

ALL_DIRS, SMALL_FILES = map_layout(STORE_TB, CFG["TB_PREFIX"])
def find_file(pred):
    hits = sorted(n for n in SMALL_FILES if pred(_norm(Path(n).name)))
    return hits[0] if hits else None
def find_dir(kind, leaf):
    hits = sorted(d for d in ALL_DIRS if _norm(Path(d.rstrip("/")).name) == leaf and kind in _norm(d))
    return hits[0] if hits else None
TB_LAYOUT = {"metadata": find_file(lambda n: n == "metadata.csv"),
             "passive_csv": find_file(lambda n: "passive" in n and n.endswith(".csv")),
             "passive_audio": find_dir("passive", "audiofiles")}
display(pd.DataFrame({"what": list(TB_LAYOUT), "found at": list(TB_LAYOUT.values())}))
assert all(TB_LAYOUT.values()), f"Not found in the tbscreen container: {[k for k, v in TB_LAYOUT.items() if not v]}"

def read_csv_blob(name):
    raw = STORE_TB.get(name)
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):        # files saved from Excel are often not UTF-8
        try:
            return pd.read_csv(io.BytesIO(raw), low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"could not decode {name}")

meta_raw = read_csv_blob(TB_LAYOUT["metadata"])
TB_CLIN_COLS = {"age": "age", "male": "gndr_a", "smoker": "smkr_a", "hiv_pos": "hiv_infected_a", "prev_tb": "prev_tb",
                "cough_days": "cough_drtn", "fever": "fever", "weight_loss": "un_wght_loss", "night_sweats": "nght_swts"}
print("Clinical columns missing from Metadata.csv:", [c for c in TB_CLIN_COLS.values() if c not in meta_raw.columns] or "none")
num = lambda c: pd.to_numeric(meta_raw.get(c), errors="coerce")
subj_col = "subject" if "subject" in meta_raw.columns else meta_raw.columns[0]
tb_clin = pd.DataFrame({"subject_id": meta_raw[subj_col].astype(str).str.strip()})
tb_clin["label"] = meta_raw["Label"].astype(str).str.strip().str.upper().map({"TB": 1, "NTB": 0, "NON-TB": 0})
tb_clin["age"] = num("age")
tb_clin["male"] = 1 - num("gndr_a")
tb_clin["smoker"] = num("smkr_a").map({0: 0, 1: 1, 2: 1})
tb_clin["hiv_pos"] = num("hiv_infected_a").map({0: 0, 1: 1})
tb_clin["prev_tb"] = num("prev_tb").map({0: 0, 1: 1, 2: 1})
tb_clin["cough_days"] = num("cough_drtn") * 7
tb_clin["fever"] = num("fever").map({0: 0, 1: 1})
tb_clin["weight_loss"] = num("un_wght_loss").map({0: 0, 1: 1})
tb_clin["night_sweats"] = num("nght_swts").map({0: 0, 1: 1})
tb_clin = tb_clin.drop_duplicates("subject_id").set_index("subject_id")
print(f"Kenya participants with a TB label: {tb_clin.label.notna().sum()} "
      f"(TB {int((tb_clin.label == 1).sum())}, non-TB {int((tb_clin.label == 0).sum())})")

DEVICES_TB = ("codec", "pixel", "yeti")
def cough_key(path, subject):
    rest = path[len(subject):] if path.startswith(subject) else path
    return subject + "|" + "_".join(t for t in rest.split("_") if t and t.lower() not in DEVICES_TB)
t = read_csv_blob(TB_LAYOUT["passive_csv"])
t["path"] = t["path"].astype(str).str.replace(r".*[\\/]", "", regex=True).str.replace(r"\.wav$", "", regex=True)
if "subject" not in t.columns:
    t["subject"] = t.path.str.extract(r"^(PID_\d+[A-Za-z]?)")[0]
if "device" not in t.columns:
    t["device"] = t.path.str.extract(r"(codec|pixel|yeti)", flags=re.I)[0]
t["subject_id"] = t.subject.astype(str).str.strip(); t["device"] = t.device.astype(str).str.lower()
t["has_audio"] = t["Permission_sound"].astype(str).str.lower().eq("yes") if "Permission_sound" in t.columns else True
t["label"] = t.subject_id.map(tb_clin.label)
print(f"Passive cough files: {len(t):,} | per device: {t.device.value_counts().to_dict()} | without a label: {t.label.isna().sum():,}")
kc = t[(t.device == CFG["KENYA_DEVICE"]) & t.has_audio & t.label.notna()].copy()
kc["label"] = kc.label.astype(int)
kc = kc.sort_values(["subject_id", "path"]).reset_index(drop=True)
kc["_draw"] = np.random.default_rng(CFG["SEED"]).random(len(kc))          # seeded random draw within each person
kc = kc.sort_values(["subject_id", "_draw"]).groupby("subject_id").head(CFG["KENYA_MAX_PER_PARTICIPANT"])
kc = kc.drop(columns="_draw").sort_values(["subject_id", "path"])
kc["blob"] = TB_LAYOUT["passive_audio"] + kc.path + ".wav"
print(f"Kenya ({CFG['KENYA_DEVICE']}): {kc.subject_id.nunique()} people, {len(kc):,} coughs to download")
got = fetch_many(STORE_TB, kc.blob.tolist(), to_clip, label="Kenya coughs")
ok = kc.blob.map(lambda n: got.get(n) is not None).values
kc = kc[ok].reset_index(drop=True)
KWAVE = np.stack([got[n][0] for n in kc.blob])
KC = pd.DataFrame({"participant": kc.subject_id.values, "label": kc.label.values, "path": kc.path.values,
                   "orig_seconds": [round(got[n][2], 3) for n in kc.blob], "orig_sr": [got[n][1] for n in kc.blob]})
KP = tb_clin.loc[tb_clin.index.isin(KC.participant), ["label"] + SHARED9].copy(); KP["label"] = KP.label.astype(int)
print(f"Kenya: {len(KP)} people ({int(KP.label.sum())} TB), {len(KC):,} clips")

## Step 5. A bigger pool of Kenyan coughs

**What we do.** Notebook 06 kept a seeded draw of up to 10 coughs per person (Step 4 above, the "original draw"). For check 1 we need other draws, so we download a larger pool: up to `KENYA_POOL_CAP` coughs per person for the **same people**, cut to the same 0.5 s clip. The other draws are then taken from this pool. No new people are added, and no Zambian label is touched.

**What to look for.** The same number of people as in Step 4 (about 118). The pool should hold more coughs than the original draw (a few thousand clips).

In [ ]:
kp = t[(t.device == CFG["KENYA_DEVICE"]) & t.has_audio & t.label.notna()].copy()
kp = kp[kp.subject_id.isin(KP.index)]                                    # the same people as the main test
kp["label"] = kp.label.astype(int)
kp = kp.sort_values(["subject_id", "path"]).reset_index(drop=True)
kp["_r"] = np.random.default_rng(CFG["SEED"] + 1000).random(len(kp))     # seeded, separate from the original draw
kp = kp.sort_values(["subject_id", "_r"]).groupby("subject_id").head(CFG["KENYA_POOL_CAP"]).drop(columns="_r")
kp = kp.sort_values(["subject_id", "path"]).reset_index(drop=True)
kp["blob"] = TB_LAYOUT["passive_audio"] + kp.path + ".wav"
print(f"Kenya pool ({CFG['KENYA_DEVICE']}): {kp.subject_id.nunique()} people, {len(kp):,} coughs to download "
      f"(original draw: {len(KC):,} coughs)")
got_p = fetch_many(STORE_TB, kp.blob.tolist(), to_clip, label="Kenya pool coughs")
ok = kp.blob.map(lambda n: got_p.get(n) is not None).values
kp = kp[ok].reset_index(drop=True)
KPWAVE = np.stack([got_p[n][0] for n in kp.blob])
KPC = pd.DataFrame({"participant": kp.subject_id.values, "label": kp.label.values, "path": kp.path.values})
assert set(KPC.participant) <= set(KP.index)
print(f"Kenya pool: {KPC.participant.nunique()} people ({int(KP.loc[KPC.participant.unique()].label.sum())} TB), {len(KPC):,} clips")
print("coughs per person in the pool: median", int(KPC.groupby('participant').size().median()),
      "| min", int(KPC.groupby('participant').size().min()), "| max", int(KPC.groupby('participant').size().max()))

## Step 6. The same three representations for every clip

**What we do.** MFCC summary (40 numbers), spectrogram summary (128) and pretrained VGGish (128), computed by the same code as Notebook 06, for the Zambia clips, the original Kenya draw and the Kenya pool. CODA is not needed here.

In [ ]:
EXTRA_ENCODERS = {}      # e.g. {"HeAR (frozen)": hear_embed}   (Week 3)

def featurize(W, name):
    X_IN = np.stack([vggish_example_padded(w, VGG_SR, seconds=1.0) for w in W]).astype(np.float32)
    F = {"MFCC summary": mfcc_summary(W),
         "Spectrogram summary": np.concatenate([X_IN.mean(axis=1), X_IN.std(axis=1)], axis=1)}
    if TORCH_OK:
        ef = CACHE / f"{name}_vggish_{len(W)}.npy"
        F["VGGish (pretrained)"] = np.load(ef) if ef.exists() else vggish_embed(X_IN)
        np.save(ef, F["VGGish (pretrained)"])
    for rep, fn in EXTRA_ENCODERS.items():
        ef = CACHE / f"{name}_{re.sub(r'[^a-z0-9]+', '_', rep.lower())}_{len(W)}.npy"
        F[rep] = np.load(ef) if ef.exists() else np.asarray(fn(W), dtype=np.float32)
        np.save(ef, F[rep])
    return F

FZ, FK, FKP = featurize(ZWAVE, "zambia"), featurize(KWAVE, "kenya"), featurize(KPWAVE, "kenya_pool")
REPS = list(FZ)
assert REPS == list(FK) == list(FKP)
print("Representations:", {k: v.shape[1] for k, v in FZ.items()})

## Step 7. Train on all of CODA, test on Zambia and on Kenya (Table E6)

**What we do.** Three models, exactly as in Notebook 04, trained on **all** CODA participants:

* **B, questions only:** regularised logistic regression on the nine shared questions.
* **Cough only:** a linear probe per clip; a person's score is the mean over their clips.
* **A, questions + cough:** B's inputs plus the cough score. The cough score used in training is an **out-of-fold** score (inner folds), so it is always a prediction for people the cough model did not see.

The **threshold for 90 % sensitivity** is fixed on CODA's out-of-fold predictions and applied unchanged to Zambia and to Kenya, as a clinic would have to do. Nothing from the test country (scaling, imputation, threshold, labels) is used in training.

**Questions that cannot be used.** A question is dropped, for both training and testing, if in the *training* dataset its missingness differs by more than 20 points between TB groups (or exceeds 50 %). The rule uses training labels only, never the test country's labels. Kenya's HIV status is the expected case. The kept questions are printed for every test set.

**Reported:** AUROC for B, cough and A with bootstrap intervals; the gain Δ = A − B with DeLong's 95 % interval on the same people; sensitivity and specificity at the frozen threshold. Each row goes into a CSV in the Appendix B format.

In [ ]:
def usable_cols(Pt, cols):
    # the same missingness rule as Notebook 04: a question whose missingness depends on TB status is a shortcut
    miss = Pt.groupby("label")[cols].apply(lambda g: g.isna().mean()).T
    gap = (miss.iloc[:, 1] - miss.iloc[:, 0]).abs()
    return [c for c in cols if gap[c] <= CFG["MAX_MISSINGNESS_GAP"] and miss.loc[c].max() < 0.5]

def sens_spec(y, flag):
    y, flag = np.asarray(y), np.asarray(flag)
    return (flag[y == 1].mean() if (y == 1).any() else np.nan), (1 - flag[y == 0].mean() if (y == 0).any() else np.nan)

def fit_external(Ptr, part_tr, X_tr, Pte, part_te, X_te, clin_cols):
    # Train B, audio and A on (Ptr, clips of Ptr) and score Pte. Thresholds come from out-of-fold predictions on the training people.
    part_tr, part_te = np.asarray(part_tr), np.asarray(part_te)
    tr = Ptr.index; te = Pte.index
    ytr = Ptr.label.values.astype(int); ylab_tr = Ptr.label.reindex(part_tr).values.astype(int)
    k = min(CFG["INNER_K"], int(pd.Series(ytr).value_counts().min()))
    a_tr = pd.Series(np.nan, index=tr)
    for itr, iva in StratifiedKFold(k, shuffle=True, random_state=CFG["SEED"]).split(tr, ytr):
        mt, mv = np.isin(part_tr, tr[itr]), np.isin(part_tr, tr[iva])
        a_tr.update(part_mean(part_tr[mv], probe(X_tr[mt], ylab_tr[mt], X_tr[mv])))
    a_te = part_mean(part_te, probe(X_tr, ylab_tr, X_te)).reindex(te)
    res = pd.DataFrame({"label": Pte.label.values.astype(int), "audio": a_te.values}, index=te)
    thr = {"audio": threshold_for(ytr, a_tr.values, CFG["SENSITIVITY_TARGET"])}
    for name in ("B", "A"):
        Xtr = Ptr[clin_cols].values if name == "B" else np.c_[Ptr[clin_cols].values, logit(a_tr.values)]
        Xte = Pte[clin_cols].values if name == "B" else np.c_[Pte[clin_cols].values, logit(a_te.values)]
        m = lr_pipe(CFG["CLIN_C"], impute=True)
        oof = cross_val_predict(m, Xtr, ytr, cv=StratifiedKFold(k, shuffle=True, random_state=CFG["SEED"]), method="predict_proba")[:, 1]
        thr[name] = threshold_for(ytr, oof, CFG["SENSITIVITY_TARGET"])
        res[name] = m.fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    for name in ("B", "audio", "A"):
        res[f"{name}_flag"] = (res[name] >= thr[name]).astype(int)
    return res

def b_rows(res, experiment, variant, representation, protocol, test_group, extra=None):
    # one row per model, in the shared results format of Appendix B
    d = delong(res.label, res.B, res.A); rows = []
    for m in ("B", "audio", "A"):
        lo, hi = boot_ci(res.label, res[m]); se, sp = sens_spec(res.label, res[f"{m}_flag"])
        row = dict(experiment=experiment, variant=variant, representation=representation, protocol=protocol,
                   test_group=test_group, model=m, n_people=len(res), n_tb=int(res.label.sum()),
                   auroc=roc_auc_score(res.label, res[m]), auroc_lo=lo, auroc_hi=hi,
                   delta=np.nan, delta_lo=np.nan, delta_hi=np.nan, p=np.nan, sens_frozen=se, spec_frozen=sp,
                   seed=CFG["SEED"], notebook_version=NB_VERSION)
        if m in ("B", "A"):
            row.update(delta=d["diff"], delta_lo=d["lo"], delta_hi=d["hi"], p=d["p"])
        rows.append({**row, **(extra or {})})
    return rows

def external_test(name, Ptr, part_tr, FX_tr, Pte, part_te, FX_te, base_cols, experiment, variant, protocol):
    # Run every representation for one (train, test) pair.
    part_te = np.asarray(part_te); Pte = Pte.loc[Pte.index.isin(part_te)]
    cols = [c for c in base_cols if c in usable_cols(Ptr, SHARED9)]          # decided on the TRAINING people only
    dropped = [c for c in base_cols if c not in cols]
    print(f"{name}: {len(Pte)} people ({int(Pte.label.sum())} TB) | questions used: {cols} | left out: {dropped or 'none'}")
    keep = np.isin(part_te, Pte.index); rows, preds = [], {}
    for rep in REPS:
        res = fit_external(Ptr, part_tr, FX_tr[rep], Pte, part_te[keep], FX_te[rep][keep], cols)
        preds[rep] = res
        rows += b_rows(res, experiment, variant, rep, protocol, name, extra=dict(questions_used=",".join(cols)))
    return rows, preds

ROWS, PRED = [], {}

## Step 8. Reproduce the Notebook 06 result

**What we do.** Train on the Kenyan people (the original draw of Step 4) and test on the Zambian people with all phones pooled: exactly the Kenya-to-Zambia run of Notebook 06. The questions kept are decided on the Kenyan training people only.

**What to look for.** The gains printed below should match Notebook 06 (about +0.028 for MFCC, +0.027 for the spectrogram, +0.013 for VGGish, within 0.002). If they do not, stop: the data differ from the earlier run and the checks below cannot be compared with it.

In [ ]:
CHK_ROWS, PRED_K = [], {}
def run_kz(name, part_tr, FX_tr, part_te, FX_te, variant, Pte=None):
    # Kenyan people train, Zambian people test
    return external_test(name, KP, part_tr, FX_tr, ZP if Pte is None else Pte, part_te, FX_te, SHARED9, "e6c", variant, "external")

rows, PRED_K["original"] = run_kz("Kenya to Zambia", KC.participant.values, FK, ZC.participant.values, FZ, "original_draw")
CHK_ROWS += rows

def gain_table(df, by=("variant", "representation")):
    out = []
    for key, d in df.groupby(list(by), sort=False):
        a, b, c = d[d.model == "A"].iloc[0], d[d.model == "B"].iloc[0], d[d.model == "audio"].iloc[0]
        out.append({**dict(zip(by, key)), "people (TB)": f"{int(a.n_people)} ({int(a.n_tb)})",
                    "AUROC questions (B)": round(b.auroc, 3), "AUROC cough": round(c.auroc, 3), "AUROC both (A)": round(a.auroc, 3),
                    "gain": round(a.delta, 3), "95% interval": f"{a.delta_lo:+.3f} to {a.delta_hi:+.3f}", "p": round(a.p, 4)})
    return pd.DataFrame(out)

CK = pd.DataFrame(CHK_ROWS)
display(gain_table(CK[CK.variant == "original_draw"]))
EXPECTED = {"MFCC summary": 0.028, "Spectrogram summary": 0.027, "VGGish (pretrained)": 0.013}
for rep, exp in EXPECTED.items():
    got_gain = CK[(CK.variant == "original_draw") & (CK.representation == rep) & (CK.model == "A")].delta.iloc[0]
    flag = "OK" if abs(got_gain - exp) <= 0.002 else "DIFFERENT: do not continue, compare with Notebook 06"
    print(f"{rep}: gain {got_gain:+.3f} (Notebook 06: {exp:+.3f})  {flag}")

## Step 9. Check 1: other draws of Kenyan coughs

**What we do.** Take `N_DRAWS` new seeded draws of up to `KENYA_MAX_PER_PARTICIPANT` coughs per person from the pool of Step 5, train on each, and test on the Zambian people (all phones pooled). One extra run uses the whole pool.

**What to look for.** The table lists, per representation, the gain across draws. A **real** effect keeps its sign and size in most draws. A **chance** effect swings around zero. Compare with the original draw of Step 8.

In [ ]:
def pool_draw(seed, cap):
    r = np.random.default_rng(seed).random(len(KPC))
    d = KPC.assign(_r=r).sort_values(["participant", "_r"]).groupby("participant").head(cap)
    return np.sort(d.index.values)

for seed in range(1, CFG["N_DRAWS"] + 1):
    idx = pool_draw(seed, CFG["KENYA_MAX_PER_PARTICIPANT"])
    rows, _ = run_kz("Kenya to Zambia", KPC.participant.values[idx], {r: FKP[r][idx] for r in REPS},
                     ZC.participant.values, FZ, f"draw_{seed}")
    CHK_ROWS += rows
allidx = np.arange(len(KPC))
rows, _ = run_kz("Kenya to Zambia", KPC.participant.values, FKP, ZC.participant.values, FZ, "whole_pool")
CHK_ROWS += rows
CK = pd.DataFrame(CHK_ROWS)

D = CK[CK.model == "A"].copy()
D1 = D[D.variant.str.startswith("draw_")]
sumrows = []
for rep, d in D1.groupby("representation", sort=False):
    c = CK[(CK.model == "audio") & CK.variant.isin(d.variant) & (CK.representation == rep)]
    sumrows.append({"representation": rep, "draws": len(d), "mean gain": round(d.delta.mean(), 3), "lowest": round(d.delta.min(), 3),
                    "highest": round(d.delta.max(), 3), "draws with gain > 0": int((d.delta > 0).sum()),
                    "draws with p < 0.05": int((d.p < 0.05).sum()), "mean cough-only AUROC": round(c.auroc.mean(), 3)})
DRAW_SUMMARY = pd.DataFrame(sumrows)
orig = D[D.variant == "original_draw"].set_index("representation").delta.round(3)
DRAW_SUMMARY["original draw gain"] = DRAW_SUMMARY.representation.map(orig)
whole = D[D.variant == "whole_pool"].set_index("representation").delta.round(3)
DRAW_SUMMARY["whole pool gain"] = DRAW_SUMMARY.representation.map(whole)
print("Gain from adding the cough, Kenya to Zambia, over other draws of Kenyan coughs:")
display(DRAW_SUMMARY)
display(gain_table(CK[CK.variant.str.startswith("draw_") | CK.variant.isin(["original_draw", "whole_pool"])],
                   by=("variant", "representation")))

## Step 10. Check 2: each Zambian phone and each site

**What we do.** Keep the Kenya-trained model of Step 8 (original draw) and score it on the Zambian people one phone at a time (A, B, C) and one site at a time. The same people appear in several phone rows, so these rows are not independent.

**What to look for.** A gain that is **general** shows up on all three phones and at both sites. A gain that sits on one phone or one site is a local effect, for example a phone-specific sound.

In [ ]:
SPLITS = [("Zambia, all phones pooled", np.ones(len(ZC), bool))]
SPLITS += [(f"Zambia, {ph}", (ZC.phone == ph).values) for ph in CFG["ZAMBIA_PHONES"]]
for s in sorted(ZP.site.dropna().unique()):
    SPLITS.append((f"Zambia site {s}", ZC.participant.isin(ZP.index[ZP.site == s]).values))
for name, mask in SPLITS:
    rows, _ = external_test(name, KP, KC.participant.values, FK, ZP, ZC.participant.values[mask],
                            {r: FZ[r][mask] for r in REPS}, SHARED9, "e6c", "kenya_train_split", "external")
    CHK_ROWS += rows
CK = pd.DataFrame(CHK_ROWS)
SPLIT_TABLE = gain_table(CK[CK.variant == "kenya_train_split"], by=("test_group", "representation"))
print("Kenya-trained model, scored on each Zambian phone and site:")
display(SPLIT_TABLE)

## Step 11. Check 3: shuffled-label control

**What we do.** Shuffle the TB labels of the Kenyan **training people** and train only the **cough model** on the shuffled labels; the questions model keeps the real labels. Score the Zambian people and compute the gain, `N_SHUFFLE` times per representation. This gives the spread of gains we would see if the cough model carried **no** label information, using the same pipeline.

**What to look for.** The observed gain (Step 8) compared with the shuffled gains. The **empirical p** is the share of shuffled runs with a gain at least as large as the observed one (with +1 in numerator and denominator). Below 0.05 means the cough score carries label information beyond what a no-signal model produces.

In [ ]:
def cough_shuffled_auc(Ptr, part_tr, X_tr, Pte, part_te, X_te, clin_cols, seed):
    # same steps as fit_external, but the cough model sees person-level labels that have been permuted
    part_tr, part_te = np.asarray(part_tr), np.asarray(part_te)
    tr, te = Ptr.index, Pte.index
    ytr = Ptr.label.values.astype(int)
    perm = pd.Series(np.random.default_rng(seed).permutation(ytr), index=tr)
    ylab_tr = perm.reindex(part_tr).values.astype(int)
    k = min(CFG["INNER_K"], int(pd.Series(ytr).value_counts().min()))
    a_tr = pd.Series(np.nan, index=tr)
    for itr, iva in StratifiedKFold(k, shuffle=True, random_state=CFG["SEED"]).split(tr, ytr):
        mt, mv = np.isin(part_tr, tr[itr]), np.isin(part_tr, tr[iva])
        a_tr.update(part_mean(part_tr[mv], probe(X_tr[mt], ylab_tr[mt], X_tr[mv])))
    a_te = part_mean(part_te, probe(X_tr, ylab_tr, X_te)).reindex(te)
    Xtr_ = np.c_[Ptr[clin_cols].values, logit(a_tr.values)]
    Xte_ = np.c_[Pte[clin_cols].values, logit(a_te.values)]
    m = lr_pipe(CFG["CLIN_C"], impute=True).fit(Xtr_, ytr)
    return roc_auc_score(Pte.label.values.astype(int), m.predict_proba(Xte_)[:, 1])

cols_kz = [c for c in SHARED9 if c in usable_cols(KP, SHARED9)]
NULL, SHUF_ROWS, t0 = {}, [], time.time()
for rep in REPS:
    res = PRED_K["original"][rep]
    assert (res.index == ZP.index).all()
    y = res.label.values
    b_auc, a_auc = roc_auc_score(y, res.B), roc_auc_score(y, res.A)
    obs = a_auc - b_auc
    null = np.array([cough_shuffled_auc(KP, KC.participant.values, FK[rep], ZP, ZC.participant.values, FZ[rep], cols_kz, seed=s) - b_auc
                     for s in range(CFG["N_SHUFFLE"])])
    NULL[rep] = null
    SHUF_ROWS.append({"representation": rep, "observed gain": round(obs, 3), "shuffled runs": len(null),
                      "shuffled mean": round(null.mean(), 3), "shuffled 95th percentile": round(np.quantile(null, 0.95), 3),
                      "shuffled max": round(null.max(), 3),
                      "empirical p": round((1 + (null >= obs).sum()) / (1 + len(null)), 3)})
    print(f"{rep}: done in {time.time() - t0:.0f} s")
SHUF_TABLE = pd.DataFrame(SHUF_ROWS)
display(SHUF_TABLE)
fig, axs = plt.subplots(1, len(REPS), figsize=(4.2 * len(REPS), 3.4), sharey=True)
for ax, rep in zip(np.atleast_1d(axs), REPS):
    ax.hist(NULL[rep], bins=20, color="#9aa7b4")
    ax.axvline(SHUF_TABLE.set_index("representation").loc[rep, "observed gain"], color="#c0392b", lw=2, label="observed")
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(rep, fontsize=10); ax.set_xlabel("gain with shuffled cough labels")
np.atleast_1d(axs)[0].set_ylabel("runs"); np.atleast_1d(axs)[0].legend()
fig.suptitle("Kenya to Zambia: observed gain against the shuffled-label spread", fontweight="bold")
fig.tight_layout(); save_fig(fig, "kz_shuffle_null"); plt.show()

## Step 12. Apply the reading rules and save the results

**What we do.** Apply the rules from the introduction to the three checks, and print one line per representation. Then write the tables, the figure and the notes to `results/e6_external/` and package them in `e6_kz_checks_results.zip`.

In [ ]:
VERD = []
for rep in REPS:
    d = CK[(CK.model == "A") & (CK.representation == rep) & CK.variant.str.startswith("draw_")]
    n_pos, n = int((d.delta > 0).sum()), len(d)
    c1 = "stable" if n_pos >= 0.8 * n else ("draw-specific" if (n - n_pos) >= n / 3 else "mixed")
    sp = CK[(CK.model == "A") & (CK.representation == rep) & (CK.variant == "kenya_train_split")]
    ph = sp[sp.test_group.str.contains("Phone|site")]
    pos = int((ph.delta > 0).sum())
    c2 = "general" if pos == len(ph) else ("absent" if pos == 0 else "local")
    pv = float(SHUF_TABLE.set_index("representation").loc[rep, "empirical p"])
    c3 = pv < 0.05
    VERD.append({"representation": rep, "check 1 (draws)": f"{c1} ({n_pos} of {n} above zero)",
                 "check 2 (phone and site)": f"{c2} ({pos} of {len(ph)} above zero)",
                 "check 3 (shuffle)": f"{'beats' if c3 else 'does not beat'} the shuffle (p = {pv})",
                 "supported": "yes" if (c1 == "stable" and c3) else "no"})
VERDICT = pd.DataFrame(VERD)
display(VERDICT)

**Save.** The cell below writes every table and the figure, and a notes file that lists the results above without interpreting them.

In [ ]:
CK = pd.DataFrame(CHK_ROWS)
CK.round(4).to_csv(RES / "kz_checks_results.csv", index=False)
DRAW_SUMMARY.to_csv(RES / "kz_check1_draws_summary.csv", index=False)
gain_table(CK[CK.variant.str.startswith("draw_") | CK.variant.isin(["original_draw", "whole_pool"])]).to_csv(RES / "kz_check1_draws_all.csv", index=False)
SPLIT_TABLE.to_csv(RES / "kz_check2_phone_site.csv", index=False)
SHUF_TABLE.to_csv(RES / "kz_check3_shuffle_summary.csv", index=False)
pd.DataFrame(NULL).round(5).to_csv(RES / "kz_check3_shuffle_values.csv", index=False)

VERDICT.to_csv(RES / "kz_checks_verdict.csv", index=False)
(RES / "notes_kz_checks.md").write_text(f"""# Experiment 6 follow-up: checks on the Kenya-to-Zambia gain

Notebook version {NB_VERSION}, seed {CFG['SEED']}.

## Configuration
* Kenya training people: {len(KP)} ({int(KP.label.sum())} TB). Zambian test people: {len(ZP)} ({int(ZP.label.sum())} TB).
* Check 1: {CFG['N_DRAWS']} other draws of up to {CFG['KENYA_MAX_PER_PARTICIPANT']} coughs per person from a pool of up to {CFG['KENYA_POOL_CAP']}.
* Check 2: each Zambian phone and each site, with the same Kenya-trained model.
* Check 3: {CFG['N_SHUFFLE']} shuffled-label runs per representation (cough model only).

## Results, by the reading rules fixed before the run
```
{VERDICT.to_string(index=False)}
```

## Check 1: gain over the draws
```
{DRAW_SUMMARY.to_string(index=False)}
```

## Check 3: observed gain against the shuffled gains
```
{SHUF_TABLE.to_string(index=False)}
```
""")

import zipfile
bundle = WORK / "export" / "e6_kz_checks_results.zip"; bundle.parent.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RES.rglob("*")):
        if f.is_file():
            z.write(f, f"results/e6_external/{f.relative_to(RES).as_posix()}")
    for f in sorted(FIG_DIR.rglob("*")):
        if f.is_file():
            z.write(f, f"figures/external/{f.relative_to(FIG_DIR).as_posix()}")
print("Saved", bundle, f"({bundle.stat().st_size / 1e6:.2f} MB). Files:")
print(sorted(f.name for f in RES.iterdir()))
print("Check: no participant IDs in the exports ->",
      not any(str(i) in (RES / 'kz_checks_results.csv').read_text() for i in list(ZP.index[:50]) + list(KP.index[:50])))